# 🥇 Gold Layer — Star Schema


## DIM

### Dim users

In [0]:
users = spark.read.table("ecom_clickstream.silver.users")
users.createOrReplaceTempView("users_silver")

In [0]:
%sql
SELECT * 
FROM users_silver

In [0]:
%sql
CREATE OR REPLACE TABLE ecom_clickstream.gold.dim_users AS
SELECT
    user_id,
    email,
    CAST(user_first_touch_timestamp AS date) AS first_touch_date,
    user_first_touch_timestamp
FROM users_silver

In [0]:
%sql
SELECT *
FROM ecom_clickstream.gold.dim_users

### Dim products

In [0]:
products = spark.read.table("ecom_clickstream.silver.products")
products.createOrReplaceTempView("silver_products")

In [0]:
%sql
SELECT *
FROM silver_products

In [0]:
%sql
CREATE OR REPLACE TABLE ecom_clickstream.gold.dim_products AS
SELECT 
    item_id,
    name AS product_name,
    price
from silver_products

In [0]:
%sql
SELECT *
FROM ecom_clickstream.gold.dim_products

## FACT sales

In [0]:
sales = spark.read.table("ecom_clickstream.silver.sales")
sales.createOrReplaceTempView("silver_sales")

In [0]:
%sql

SELECT *
FROM silver_sales

In [0]:
%sql
CREATE OR REPLACE TABLE ecom_clickstream.gold.fact_sales AS
SELECT 
    ss.order_id,
    du.user_id,
    ss.item_id,
    ss.coupon,
    ss.transaction_timestamp,
    ss.purchase_revenue_in_usd,
    ss.item_revenue_in_usd,
    ss.price_in_usd,
    ss.quantity,
    ss.total_item_quantity,
    ss.unique_items
FROM silver_sales AS ss
LEFT JOIN ecom_clickstream.gold.dim_users AS du
ON ss.email = du.email


##FACT events

In [0]:
events = spark.read.table("ecom_clickstream.silver.events")
events.createOrReplaceTempView("silver_events")

In [0]:
%sql
CREATE OR REPLACE TABLE ecom_clickstream.gold.fact_event AS
SELECT
user_id,
device,
event_name,
traffic_source,
event_previous_timestamp,
event_timestamp,
user_first_touch_timestamp,
city,
state
FROM silver_events

In [0]:
%sql
SELECT *
FROM ecom_clickstream.gold.fact_event